In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import h5netcdf
import matplotlib.colors as mcolors
import pandas as pd
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
path = r"insert historical dataset path"
ds = xr.open_dataset(path)
path2 = r"insert future dataset path"
ds_2 = xr.open_dataset(path2)

colors:

major wet: darkorchid

minor wet: violet

major dry: goldenrod

minor dry: golden

major hot: mediumvioletred

minor hot: palevioletred

major cold: darkblue

minor cold: cornflowerblue

In [ ]:
# specify all transitions (ones that actually occur)
zone_ids = {'A->xAr': 1, 'B->xAr':2, 'C->xAr':3, 'B->A':4, 'C->A':5, 'A->B':6, 'C->B':7, 'D->B':8, 'E->B':9,
            'B->C':10, 'D->C':11, 'F->C':12, 'E->D':13, 'F->D':14, 'F->E': 15, 'BWh->BSh': 16, 'BWk->BWh': 17, 
            'BWk->BSh': 18, 'BWk->BSk': 19, 'BSh->BWh': 20, 'BSh->BSk': 21, 'BSk->BWh': 22,'BSk->BWk': 23, 
            'BSk->BSh': 24, 'Ar->Aw': 25, 'Aw->Ar': 26, 'Cf->Cm': 27, 'Cm->Cf': 28,'DC->DO': 29, 'DO->DC': 30, 
            'Fi->Ft': 31, 'Ar->xAr':32, 'Ar->xAw': 33, 'Aw->xAw': 34, 'Aw->xAr': 35, 'BSh->xAr': 36, 'BSh->xAw': 37,
           'BWh->xAr':38, 'BWh->xAw': 39, 'Cf->xAr': 40, 'Cm->xAr': 41, 'Cm->xAr': 42, 'Cm->xAw': 43, 'BSh->BSk':44,
           'Ft->Fi': 45, 'BWh->BSk': 46, 'BSh->BWk': 47} 

# color-code based on type of transition
zone_colors = {'A->xAr': 'mediumvioletred', 'B->xAr':'darkorchid', 'C->xAr':'mediumvioletred', 'B->A':'darkorchid', 
               'C->A':'mediumvioletred', 'A->B':'goldenrod', 'C->B':'goldenrod', 'D->B':'goldenrod', 'E->B':'goldenrod', 
               'B->C':'darkorchid', 'D->C':'mediumvioletred', 'F->C':'mediumvioletred', 'E->D':'mediumvioletred', 'F->D':'mediumvioletred', 
               'F->E':'mediumvioletred', 'BWh->BSh': 'violet', 'BWk->BWh': 'palevioletred', 'BWk->BSh': 'violet', 'BWk->BSk': 'violet', 
               'BSh->BWh': 'gold','BSh->BSk': 'cornflowerblue', 'BSk->BWh': 'gold','BSk->BWk': 'gold', 'BSk->BSh': 'palevioletred', 
               'Ar->Aw': 'gold', 'Aw->Ar': 'violet', 'Cf->Cm': 'gold', 'Cm->Cf': 'violet','DC->DO': 'palevioletred', 'DO->DC': 'cornflowerblue', 
               'Fi->Ft': 'palevioletred', 'Ar->xAr': 'mediumvioletred', 'Ar->xAw': 'mediumvioletred', 'Aw->xAw': 'mediumvioletred', 'Aw->xAr': 'mediumvioletred',
               'BSh->xAr': 'darkorchid', 'BSh->xAw': 'darkorchid','BWh->xAr':'darkorchid', 'BWh->xAw': 'darkorchid', 'Cf->xAr': 'mediumvioletred', 
               'Cm->xAr': 'mediumvioletred', 'Cm->xAr': 'mediumvioletred', 'Cm->xAw': 'mediumvioletred', 'BSh->BSk': 'cornflowerblue', 'Ft->Fi': 'cornflowerblue',
              'BWh->BSk': 'violet', 'BSh->BWk': 'gold'}

In [ ]:
#major
zones_hist = {
    'A': ds['A'],
    'B': ds['B'],
    'C': ds['C'],
    'D': ds['D'],
    'E': ds['E'],
    'F': ds['F']
}

zones_future = {
    'xAr': ds_2['xAr'],
    'xAw': ds_2['xAw'],
    'A':  ds_2['A'],
    'B':  ds_2['B'],
    'C':  ds_2['C'],
    'D':  ds_2['D'],
    'E':  ds_2['E'],
    'F':  ds_2['F']
}

In [ ]:
#minor
zones_hist_minor = {
    'BWh': ds['BWh'],
    'BWk': ds['BWk'],
    'BSh': ds['BSh'],
    'BSk': ds['BSk'],
    'Ar': ds['Ar'],
    'Aw': ds['Aw'],
    'Cf': ds['Cf'],
    'Cm': ds['Cm'],
    'DO': ds['DO'],
    'DC': ds['DC'],
    'Ft': ds['Ft'],
    'Fi': ds['Fi']
}

zones_future_minor = {
    'BWh': ds_2['BWh'],
    'BWk': ds_2['BWk'],
    'BSh': ds_2['BSh'],
    'BSk': ds_2['BSk'],
    'Ar': ds_2['Ar'],
    'Aw': ds_2['Aw'],
    'Cf': ds_2['Cf'],
    'Cm': ds_2['Cm'],
    'DO': ds_2['DO'],
    'DC': ds_2['DC'],
    'Ft': ds_2['Ft'],
    'Fi': ds_2['Fi']
}

In [ ]:
# combine
hist_zones = zones_hist | zones_hist_minor
fut_zones = zones_future | zones_future_minor

In [ ]:
# function creating labels assigned to each grid cell
def collapse_zone_masks(zone_dict, zone_order):
    first_key = list(zone_dict.keys())[0]
    shape = zone_dict[first_key].shape
    out = np.full(shape, '', dtype=object)

    keys = zone_order if zone_order else list(zone_dict.keys())

    for label in keys:
        mask = zone_dict[label].values.astype(bool)
        print(f"{label}: {mask.sum()} cells → writing '{label}'")
        out[mask] = label

    print("Total labeled cells:", np.count_nonzero(out != ''))
    return out # check

In [ ]:
zone_order_hist = ["A", "B", "C", "D", "E", "F", "Ar", "Aw", "BWh", "BWk", "BSh", "BSk",
                  "Cf", "Cm", "DO", "DC", "Ft", "Fi"]
historical_zones = collapse_zone_masks(hist_zones, zone_order=zone_order_hist)

zone_order_fut = zone_order_hist + ['xAw', 'xAr']
future_zones = collapse_zone_masks(fut_zones, zone_order=zone_order_fut) # check

In [ ]:
np.unique(future_zones) # check

In [ ]:
hist_str = historical_zones.astype(str)
fut_str = future_zones.astype(str)

# make the labels "pretty" if you want
transitions = np.char.add(np.char.add(hist_str, "→"), fut_str)
transitions_fixed = np.char.replace(transitions, '→', '->')

In [ ]:
flat_trans = transitions_fixed.flatten()
unique_trans = pd.unique(flat_trans) # check
trans_int = {label: i for i, label in enumerate(unique_trans)}

# make 2D array of integers representing transitions
int_map = np.vectorize(trans_int.get)(transitions_fixed)

In [ ]:
# map subtypes back to their major categories
subtype_to_major = {
    "Cf": "C", "Cm": "C",
    "BWk": "B", "BWh": "B", "BSh": "B", "BSk": "B",
    "Ar": "A", "Aw": "A",
    "DC": "D", "DO": "D",
    "Fi": "F", "Ft": "F",
    "E": "E",
    "xAw": "xA", "xAr": "xA"
    # add others if needed
}

In [ ]:
# minor transitions (already subtype vs subtype)
transitions_minor = np.char.add(
    np.char.add(historical_zones.astype(str), "->"),
    future_zones.astype(str)
)

# major transitions (subtype mapped to parent)
historical_major = np.vectorize(subtype_to_major.get)(historical_zones)
future_major     = np.vectorize(subtype_to_major.get)(future_zones)

transitions_major = np.char.add(
    np.char.add(historical_major, "->"),
    future_major
)

In [ ]:
# stack both sets together
combined_transitions = {
    "major": transitions_major,
    "minor": transitions_minor
}


# build ID maps by assigning climate zones numbers --> make numeric array
major_ids = np.full(transitions_major.shape, np.nan)
for label, idx in zone_ids.items():
    major_ids[transitions_major == label] = idx

minor_ids = np.full(transitions_minor.shape, np.nan)
for label, idx in zone_ids.items():
    minor_ids[transitions_minor == label] = idx

# combine: majors take priority, minors fill in where majors aren't defined
id_grid = np.where(~np.isnan(major_ids), major_ids, minor_ids)

In [ ]:
np.unique(id_grid) # check

In [ ]:
color_grid = np.full(id_grid.shape, 'white', dtype=object)
for label, idx in zone_ids.items(): # convert numeric IDs to colors
    color_grid[id_grid == idx] = zone_colors[label]

In [ ]:
lon = ds['lon']
lat = ds['lat']

# create map figure

# convert color strings to RGB for plotting
rgb_array = np.vectorize(mcolors.to_rgb)(color_grid)
rgb_image = np.moveaxis(rgb_array, 0, -1)

# set up map projection
proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=proj)

# background features
ax.set_global()
ax.coastlines(resolution="110m", linewidth=0.7)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="none")
ax.add_feature(cfeature.OCEAN, facecolor="none", edgecolor="none")

# gridlines
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                  color="teal", linewidth=0.4)
gl.xlabel_style = {"size": 10, "color": "teal"}
gl.ylabel_style = {"size": 10, "color": "teal"}

# overlay transitions
extent = [float(lon.min()), float(lon.max()), float(lat.min()), float(lat.max())]
ax.imshow(
    rgb_image,
    origin="lower",
    extent=extent,
    transform=ccrs.PlateCarree(),
    interpolation="none"
)

# title
ax.set_title("insert title here", fontsize=18, fontname="Times New Roman")

# save fig
plt.savefig('insert figure path here', dpi=300, bbox_inches='tight')
plt.show()